# Water Phantom Profiles from Commissioning Data

This notebook compares commissioning measurements against PyDoseRT simulation for 10 MV water phantom beams.

Figure layout:
- Top-left: depth-dose curves (Ref vs PyDoseRT) for 5x5, 10x10, 20x20 cm$^2$
- Top-right: lateral profiles at selected depth (Ref vs PyDoseRT)
- Bottom-left: depth-dose difference (PyDoseRT - Ref)
- Bottom-right: lateral-profile difference (PyDoseRT - Ref)


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    candidates = [start] + list(start.parents)
    for c in candidates:
        if (c / 'commissioning').exists() and (c / 'src').exists():
            return c
    raise FileNotFoundError('Could not locate repository root containing commissioning/ and src/.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import torch
import pydosert as PDRT
from commissioning.toolkit.commissioning_parser import MeasurementParser

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print('device:', device)
print('REPO_ROOT:', REPO_ROOT)


In [ ]:
# -------------------------------
# User controls
# -------------------------------
DD_FILE = REPO_ROOT / 'commissioning/data/measurements_10MV/measurements_10_dd.asc'
PROFILES_FILE = REPO_ROOT / 'commissioning/data/measurements_10MV/measurements_10_profiles.asc'

FIELD_SIZES_MM = [50.0, 100.0, 200.0]  # 5x5, 10x10, 20x20 cm^2
LATERAL_AXIS = 'Y'                      # commissioning profile axis: 'X' or 'Y'
LATERAL_DEPTH_MM = 100.0                # commissioning profile depth (z in .asc), mm
LATERAL_RANGE_MM = (-150.0, 150.0)

# Phantom/model setup
PHANTOM_SHAPE = (185, 167, 167)
PHANTOM_SPACING_MM = (3.0, 3.0, 3.0)
ISO_CENTER_MM = (277.5, 100.0, 251.0)

# Commissioned machine config
MACHINE_CONFIG_JSON = REPO_ROOT / 'commissioning/machine_config_10MV_VersaHD.json'
machine_config = PDRT.MachineConfig(preset=str(MACHINE_CONFIG_JSON))

print('DD_FILE:', DD_FILE)
print('PROFILES_FILE:', PROFILES_FILE)
print('MACHINE_CONFIG_JSON:', MACHINE_CONFIG_JSON)




In [ ]:
if not DD_FILE.exists():
    raise FileNotFoundError(f'Missing depth-dose file: {DD_FILE}')
if not PROFILES_FILE.exists():
    raise FileNotFoundError(f'Missing profile file: {PROFILES_FILE}')

dd_profiles = MeasurementParser.parse_rfa300(str(DD_FILE))
lat_profiles = MeasurementParser.parse_rfa300(str(PROFILES_FILE))

def _close_fs(fs_a, fs_b, tol_mm=2.0):
    return abs(fs_a[0] - fs_b[0]) <= tol_mm and abs(fs_a[1] - fs_b[1]) <= tol_mm

def select_pdd_profile(profiles, field_size_mm):
    target = (float(field_size_mm), float(field_size_mm))
    candidates = [p for p in profiles if p.scan_type == 'PDD' and _close_fs(p.field_size_mm, target)]
    if not candidates:
        return None
    return candidates[0]

def select_lateral_profile(profiles, field_size_mm, depth_mm, axis='Y'):
    target = (float(field_size_mm), float(field_size_mm))
    candidates = [
        p for p in profiles
        if p.scan_type == 'PRO'
        and p.axis.upper() == axis.upper()
        and p.depth_mm is not None
        and abs(float(p.depth_mm) - float(depth_mm)) <= 1.0
        and _close_fs(p.field_size_mm, target)
    ]
    if not candidates:
        return None
    return candidates[0]

def sort_curve(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    idx = np.argsort(x)
    return x[idx], y[idx]

def normalize_100(y):
    y = np.asarray(y, dtype=float)
    m = float(np.max(y)) if y.size else 0.0
    if m <= 0.0:
        return y
    return y / m * 100.0

print('Loaded commissioning profiles.')
print('  DD profiles :', len(dd_profiles))
print('  Lateral profiles:', len(lat_profiles))


In [ ]:
phantom = PDRT.Phantom.from_uniform_water(shape=PHANTOM_SHAPE, spacing=PHANTOM_SPACING_MM)
engine = PDRT.DoseEngine(
    machine_config=machine_config,
    dose_grid_spacing=phantom.resolution,
    dose_grid_shape=PHANTOM_SHAPE,
    kernel_size=151,
    device=device,
    dtype=dtype,
)
phantom = phantom.to(engine.dtype).to(engine.device)

spacing = tuple(float(x) for x in engine.dose_grid_spacing)
shape = tuple(int(x) for x in engine.dose_grid_shape)

# Use physical isocenter voxel for central-axis sampling (critical for small fields like 5x5).
iso_x_idx = int(round(float(ISO_CENTER_MM[0]) / spacing[0]))
iso_y_idx = int(round(float(ISO_CENTER_MM[1]) / spacing[1]))
iso_z_idx = int(round(float(ISO_CENTER_MM[2]) / spacing[2]))
iso_x_idx = max(0, min(shape[0]-1, iso_x_idx))
iso_y_idx = max(0, min(shape[1]-1, iso_y_idx))
iso_z_idx = max(0, min(shape[2]-1, iso_z_idx))

# Lateral profile depth index from user depth in mm
lat_depth_idx = int(round(float(LATERAL_DEPTH_MM) / spacing[1]))
lat_depth_idx = max(0, min(shape[1] - 1, lat_depth_idx))

axis_key = LATERAL_AXIS.upper()
if axis_key == 'Y':
    # Commissioning Y profile maps best to model axis-0 in this phantom setup
    lat_dim = 0
    lat_center_idx = iso_x_idx
    lat_spacing = spacing[0]
    def extract_lateral_curve(dose_arr, start_idx, end_idx):
        return dose_arr[start_idx:end_idx, lat_depth_idx, iso_z_idx]
elif axis_key == 'X':
    # Commissioning X profile maps to model axis-2 in this phantom setup
    lat_dim = 2
    lat_center_idx = iso_z_idx
    lat_spacing = spacing[2]
    def extract_lateral_curve(dose_arr, start_idx, end_idx):
        return dose_arr[iso_x_idx, lat_depth_idx, start_idx:end_idx]
else:
    raise ValueError(f'Unsupported LATERAL_AXIS={LATERAL_AXIS!r}. Use X or Y.')

lat_half_idx = int(round(abs(LATERAL_RANGE_MM[1]) / lat_spacing))
lat_dim_size = shape[lat_dim]
lat_start = max(0, lat_center_idx - lat_half_idx)
lat_end = min(lat_dim_size, lat_center_idx + lat_half_idx + 1)
lat_indices = np.arange(lat_start, lat_end)
lat_axis_mm = (lat_indices - lat_center_idx) * lat_spacing

# Depth axis = model axis-1 (physical y direction)
depth_axis_mm_pred = np.arange(shape[1]) * spacing[1]

results = {}
for fs_mm in FIELD_SIZES_MM:
    beam = PDRT.Beam.create(
        gantry_angle_deg=0.0,
        number_of_leaf_pairs=int(machine_config.number_of_leaf_pairs),
        field_size_mm=(float(fs_mm), float(fs_mm)),
        iso_center=ISO_CENTER_MM,
        device=engine.device,
        dtype=engine.dtype,
    )

    dose_pred = engine.compute_dose(beam, density_image=phantom.density_image).detach().cpu().numpy()[0]

    # Model depth-dose on central axis
    depth_curve_pred = dose_pred[iso_x_idx, :, iso_z_idx]

    # Model lateral profile at requested depth
    lat_curve_pred = extract_lateral_curve(dose_pred, lat_start, lat_end)

    pdd_ref = select_pdd_profile(dd_profiles, fs_mm)
    lat_ref = select_lateral_profile(lat_profiles, fs_mm, LATERAL_DEPTH_MM, axis=LATERAL_AXIS)

    if pdd_ref is None:
        raise ValueError(f'No PDD reference found for field {fs_mm} mm')
    if lat_ref is None:
        raise ValueError(f'No lateral reference found for field {fs_mm} mm at depth {LATERAL_DEPTH_MM} mm axis {LATERAL_AXIS}')

    pdd_x_mm_ref, pdd_y_ref_raw = sort_curve(pdd_ref.position_mm, pdd_ref.dose_values)
    lat_x_mm_ref, lat_y_ref_raw = sort_curve(lat_ref.position_mm, lat_ref.dose_values)

    # Normalize each curve to 100 for shape comparison
    pdd_y_ref = normalize_100(pdd_y_ref_raw)
    lat_y_ref = normalize_100(lat_y_ref_raw)
    depth_curve_pred_n = normalize_100(depth_curve_pred)
    lat_curve_pred_n = normalize_100(lat_curve_pred)

    # Interpolate model onto commissioning measurement positions
    pdd_y_pred = np.interp(pdd_x_mm_ref, depth_axis_mm_pred, depth_curve_pred_n)
    lat_y_pred = np.interp(lat_x_mm_ref, lat_axis_mm, lat_curve_pred_n, left=np.nan, right=np.nan)

    pdd_diff = pdd_y_pred - pdd_y_ref
    lat_diff = lat_y_pred - lat_y_ref

    results[fs_mm] = {
        'pdd_x_ref': pdd_x_mm_ref,
        'pdd_ref': pdd_y_ref,
        'pdd_pred': pdd_y_pred,
        'pdd_diff': pdd_diff,
        'lat_x_ref': lat_x_mm_ref,
        'lat_ref': lat_y_ref,
        'lat_pred': lat_y_pred,
        'lat_diff': lat_diff,
    }

print('Simulation + reference extraction complete.')
print('Isocenter voxel [x,y,z]:', (iso_x_idx, iso_y_idx, iso_z_idx))
print('Lateral model extraction axis:', axis_key, 'dimension=', lat_dim)
print('Lateral model extraction depth index (axis-1):', lat_depth_idx, '->', lat_depth_idx * spacing[1], 'mm')
print('Lateral model extraction range [mm]:', (float(lat_axis_mm.min()), float(lat_axis_mm.max())))



In [ ]:
field_labels = {50.0: '5x5 cm$^2$', 100.0: '10x10 cm$^2$', 200.0: '20x20 cm$^2$'}
colors = {50.0: '#1f77b4', 100.0: '#2ca02c', 200.0: '#d62728'}

# Typography tuned for publication readability on a large 2x2 layout
FONT_BASE = 16
FONT_TICK = 14
FONT_LABEL = 16
FONT_TITLE = 19
FONT_LEGEND = 13

plt.rcParams.update({
    'font.size': FONT_BASE,
    'axes.titlesize': FONT_TITLE,
    'axes.labelsize': FONT_LABEL,
    'xtick.labelsize': FONT_TICK,
    'ytick.labelsize': FONT_TICK,
    'legend.fontsize': FONT_LEGEND,
})

# Share x-axis within each column and remove vertical gap between top/bottom panels
fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 11),
    sharex='col',
    constrained_layout=False,
    gridspec_kw={'height_ratios': [2, 1]}
)
fig.subplots_adjust(hspace=0.0, wspace=0.22)

ax_depth = axes[0, 0]
ax_lat = axes[0, 1]
ax_depth_diff = axes[1, 0]
ax_lat_diff = axes[1, 1]

for fs_mm in FIELD_SIZES_MM:
    c = colors[fs_mm]
    lbl = field_labels[fs_mm]
    r = results[fs_mm]

    # Top-left: depth dose
    ax_depth.plot(r['pdd_x_ref'], r['pdd_ref'], '--', color=c, linewidth=2.2, label=f'{lbl} Reference (TPS)')
    ax_depth.plot(r['pdd_x_ref'], r['pdd_pred'], '-', color=c, linewidth=2.5, label=f'{lbl} PyDoseRT')

    # Top-right: lateral profiles
    ax_lat.plot(r['lat_x_ref'], r['lat_ref'], '--', color=c, linewidth=2.2, label=f'{lbl} Reference (TPS)')
    ax_lat.plot(r['lat_x_ref'], r['lat_pred'], '-', color=c, linewidth=2.5, label=f'{lbl} PyDoseRT')

    # Bottom-left: depth differences
    ax_depth_diff.plot(r['pdd_x_ref'], r['pdd_diff'], '-', color=c, linewidth=2.2, label=lbl)

    # Bottom-right: lateral differences
    valid = np.isfinite(r['lat_diff'])
    ax_lat_diff.plot(r['lat_x_ref'][valid], r['lat_diff'][valid], '-', color=c, linewidth=2.2, label=lbl)

ax_depth.set_title('Depth-dose curves')
ax_depth.set_ylabel('Relative dose [%]')
ax_depth.grid(True, linestyle=':', alpha=0.5)
ax_depth.legend(loc='upper right', ncol=1)
ax_depth.tick_params(axis='x', labelbottom=False)

ax_lat.set_title(f'Lateral profiles at depth {LATERAL_DEPTH_MM:.0f} mm (axis {LATERAL_AXIS})')
ax_lat.set_ylabel('Relative dose [%]')
ax_lat.set_xlim(*LATERAL_RANGE_MM)
ax_lat.set_ylim(0.0, 120.0)
ax_lat.grid(True, linestyle=':', alpha=0.5)
ax_lat.tick_params(axis='x', labelbottom=False)

# ax_depth_diff.set_title('Depth-dose difference (PyDoseRT - Ref)')
ax_depth_diff.set_xlabel('Depth [mm]')
ax_depth_diff.set_ylabel('Difference [%]')
ax_depth_diff.axhline(0.0, color='black', linewidth=1.2, alpha=0.8)
ax_depth_diff.grid(True, linestyle=':', alpha=0.5)
ax_depth_diff.legend(loc='upper right', ncol=1)

# ax_lat_diff.set_title('Lateral difference (PyDoseRT - Ref)')
ax_lat_diff.set_xlabel('Lateral position [mm]')
ax_lat_diff.set_ylabel('Difference [%]')
ax_lat_diff.set_xlim(*LATERAL_RANGE_MM)
ax_lat_diff.axhline(0.0, color='black', linewidth=1.2, alpha=0.8)
ax_lat_diff.grid(True, linestyle=':', alpha=0.5)

for ax in [ax_depth, ax_lat, ax_depth_diff, ax_lat_diff]:
    ax.tick_params(axis='both', which='both', labelsize=FONT_TICK)

plt.show()
